# Analytics and serving with FeatureMesh

This Colab notebook was generated from the FeatureMesh docs tutorial.

Work top to bottom: first install FeatureMesh plus local Redis, then create clients, then load data and run the tutorial.

1. **Install infrastructure** — Python packages, start local services, smoke-test them.
2. **Create FeatureMesh clients** — Jupyter magic, `ServingClient`, and serving executors.
3. **Set up tutorial data**, then follow the tutorial sections in order.


## 1. Install infrastructure

Install the FeatureMesh client, start Redis in this Colab VM, and verify a write/read round trip before creating clients.


In [ ]:
%pip install -q "featuremesh[serving]" pandas redis


In [ ]:
!apt-get -qq update
!apt-get install -y redis-server
!redis-server --daemonize yes


In [ ]:
import time

time.sleep(1)

# Redis: verify connection and scalar/list round trips.
import redis
r = redis.Redis(host="127.0.0.1", port=6379, decode_responses=True)
print("Redis ping:", r.ping())
r.set("smoke_test", "hello", ex=60)
print("Redis read back:", r.get("smoke_test"))
r.delete("smoke_test")

r.rpush("smoke_list", "a", "b")
print("Redis list:", r.lrange("smoke_list", 0, -1))
r.delete("smoke_list")
print("Redis OK")


## 2. Create FeatureMesh clients

Load the Jupyter magic, create a local `ServingClient`, and wire the serving executors.


In [ ]:
%load_ext featuremesh


In [ ]:
from IPython.display import display
from featuremesh import RegistryDeployment, ServingClient, ServingDeployment, set_default
from featuremesh.helpers.sltest_executors import RedisSltExecutor, SqlSltExecutor
from redis import Redis

set_default("registry", RegistryDeployment.LOCAL)
set_default("serving", ServingDeployment.EMBEDDED)

serving_client = ServingClient()
set_default("client", serving_client)

SERVING_EXECUTORS = {}
SERVING_EXECUTORS["redis"] = RedisSltExecutor(
    Redis.from_url("redis://127.0.0.1:6379/0", decode_responses=True)
)

from featuremesh import BatchClient

client = BatchClient()
print("FeatureMesh clients ready; serving executors: Redis")


This page is the interactive companion to the FeatureMesh homepage [How it works](https://featuremesh.com/#how-it-works) walkthrough. Same path, runnable end to end: namespace `fm.home`, a tiny customer/orders dataset, one promo rule (`show_promocode`), batch on the warehouse, then the same rule from Redis via `VARIANT()` for serving.

This is the product walkthrough, not the language introduction. If `ENTITY()`, `FOR`, bindings, or `RELATED()` are new, start with the [FeatureQL companion](https://featuremesh.com/docs/tutorials/homepage/featureql), then return here.


## Set up the analytics data

Three tables: customers, orders, and a small OBT with each customer's last order and order-id array. Edit the `VALUES` if you want different demo data.


### Customers


In [3]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE SCHEMA IF NOT EXISTS home;


In [4]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE OR REPLACE TABLE home.dim_customers AS
SELECT
    customer_id::BIGINT AS customer_id,
    name,
    created_at
FROM (
    VALUES
        (100, 'Alice', DATE '2022-03-15'),
        (101, 'Bob', DATE '2023-06-10'),
        (102, 'Charlie', DATE '2024-01-20'),
        (103, 'Diana', DATE '2024-11-01')
) AS t(customer_id, name, created_at);


### Orders


In [5]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE OR REPLACE TABLE home.fct_orders AS
SELECT
    order_id::BIGINT AS order_id,
    order_customer_id::BIGINT AS order_customer_id,
    price::DECIMAL(10, 2) AS price,
    created_at
FROM (
    VALUES
        (1001, 100, 450.00, TIMESTAMP '2025-06-01 08:00:00'),
        (1002, 100, 380.00, TIMESTAMP '2025-07-15 14:45:00'),
        (1003, 101, 600.00, TIMESTAMP '2025-08-20 17:30:00'),
        (1004, 101, 550.00, TIMESTAMP '2025-10-05 16:20:00'),
        (1005, 100, 520.00, TIMESTAMP '2025-11-12 11:30:00'),
        (1006, 102, 1200.00, TIMESTAMP '2025-11-15 14:30:00'),
        (1007, 102, 300.00, TIMESTAMP '2025-12-20 10:00:00'),
        (1008, 101, 400.00, TIMESTAMP '2025-12-25 09:45:00'),
        (1009, 103, 850.00, TIMESTAMP '2026-01-10 12:00:00'),
        (1010, 103, 400.00, TIMESTAMP '2026-01-14 09:30:00')
) AS t(order_id, order_customer_id, price, created_at);


### Customer OBT


In [6]:
%%featureql --client client --hide-dataframe

/* SQL */
CREATE OR REPLACE TABLE home.agg_customers_obt AS
SELECT
    customer_id::BIGINT AS customer_id,
    last_order_id::BIGINT AS last_order_id,
    orders::BIGINT[] AS orders
FROM (
    VALUES
        (100, 1005, [1001, 1002, 1005]),
        (101, 1008, [1003, 1004, 1008]),
        (102, 1007, [1006, 1007]),
        (103, 1010, [1009, 1010])
) AS t(customer_id, last_order_id, orders);


## Analytics / Training

### Define entities and keys

Entities are the semantic foundation: `customers` and `orders` are business objects, not tables. Their keys are typed — `BIGINT#customers` is not interchangeable with `BIGINT#orders` — so invalid joins fail at compile time instead of in a dashboard review.


In [7]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN fm.home UP TO LEVEL 9;


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.HOME UP TO LEVEL 9) (acknowledge with ACK-V2MG)


In [8]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN fm.home WHERE FUNCTION = 'KEYSET'


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.HOME WHERE FUNCTION = 'KEYSET') (acknowledge with ACK-53KA)


In [9]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.HOME AS
SELECT
    -- Entities
    customers := ENTITY(),
    orders := ENTITY(),
    -- Primary keys
    customer_id := INPUT(BIGINT#customers),
    order_id := INPUT(BIGINT#orders),
    -- Customer dimensions
    tables.dim_customers := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        name VARCHAR,
        created_at TIMESTAMP
        FROM TABLE(home.dim_customers)
    ),
    customer_name := tables.dim_customers[name],
    customer_created_at := tables.dim_customers[created_at],
    -- Order facts
    tables.fct_orders := EXTERNAL_COLUMNS(
        order_id BIGINT#orders BIND TO order_id,
        order_customer_id BIGINT#customers,
        price DECIMAL,
        created_at TIMESTAMP
        FROM TABLE(home.fct_orders)
    ),
    order_price := tables.fct_orders[price],
    order_customer_id := tables.fct_orders[order_customer_id],
    order_created_at := tables.fct_orders[created_at],
    -- Customer facts aggregations (OBT)
    tables.agg_customers_obt := EXTERNAL_COLUMNS(
        customer_id BIGINT#customers BIND TO customer_id,
        last_order_id BIGINT#orders,
        orders ARRAY(BIGINT#orders)
        FROM TABLE(home.agg_customers_obt)
    ),
    last_order_id := tables.agg_customers_obt[last_order_id],
    customer_orders := tables.agg_customers_obt[orders],
    -- Keysets: Define where to find the keys for each entity.
    dim_customers_keyset := KEYSET(
        customers,
        'SELECT customer_id AS "fm.home.customer_id"
         FROM home.dim_customers'
    ),
    fct_orders_keyset := KEYSET(
        orders,
        'SELECT order_id AS "fm.home.order_id"
         FROM home.fct_orders'
    )
;


,feature_name,status,message
0,FM.HOME.CUSTOMERS,CREATED,Feature created as not exists
1,FM.HOME.ORDERS,CREATED,Feature created as not exists
2,FM.HOME.CUSTOMER_ID,CREATED,Feature created as not exists
3,FM.HOME.ORDER_ID,CREATED,Feature created as not exists
4,FM.HOME.TABLES.DIM_CUSTOMERS,CREATED,Feature created as not exists
5,FM.HOME.CUSTOMER_NAME,CREATED,Feature created as not exists
6,FM.HOME.CUSTOMER_CREATED_AT,CREATED,Feature created as not exists
7,FM.HOME.TABLES.FCT_ORDERS,CREATED,Feature created as not exists
8,FM.HOME.ORDER_PRICE,CREATED,Feature created as not exists
9,FM.HOME.ORDER_CUSTOMER_ID,CREATED,Feature created as not exists


The `KEYSET()` definitions at the bottom are how you enumerate “all customers” later — pin the keyset feature with `@BIND_KEYSET(dim_customers_keyset)`, without hard-coding ids in every query.

### Map features to columns

In the same statement, `EXTERNAL_COLUMNS()` turns warehouse columns into source features (`order_price`, `customer_orders`, `last_order_id`, …). `BIND TO` says which column is the entity key for that table.

Derived names stay separate from `tables.*` mappings on purpose: business logic composes on feature names, not on physical schemas. Swap the table tomorrow; keep the formulas.

### Write transformations

Express the business decision as features — not as a one-off SQL report. Persist lifetime value, recency, and the batch promo rule. Lifetime value is also stored in cents so the same predicate can be reused for serving without floating-point compares.


In [10]:
%%featureql --client serving_client --hide-dataframe

/* SQL */
CREATE SCHEMA IF NOT EXISTS home;
--
DROP TABLE IF EXISTS home.dim_customers;
--
DROP TABLE IF EXISTS home.fct_orders;
--
DROP TABLE IF EXISTS home.agg_customers_obt;
--
CREATE TABLE home.dim_customers (
  customer_id BIGINT,
  name VARCHAR,
  created_at DATE
);
--
INSERT INTO home.dim_customers VALUES
  (CAST(100 AS BIGINT), 'Alice', DATE '2022-03-15'),
  (CAST(101 AS BIGINT), 'Bob', DATE '2023-06-10'),
  (CAST(102 AS BIGINT), 'Charlie', DATE '2024-01-20'),
  (CAST(103 AS BIGINT), 'Diana', DATE '2024-11-01');
--
CREATE TABLE home.fct_orders (
  order_id BIGINT,
  order_customer_id BIGINT,
  price DECIMAL(10, 2),
  created_at TIMESTAMP
);
--
INSERT INTO home.fct_orders VALUES
  (CAST(1001 AS BIGINT), CAST(100 AS BIGINT), CAST(450.00 AS DECIMAL(10,2)), TIMESTAMP '2025-06-01 08:00:00'),
  (CAST(1002 AS BIGINT), CAST(100 AS BIGINT), CAST(380.00 AS DECIMAL(10,2)), TIMESTAMP '2025-07-15 14:45:00'),
  (CAST(1003 AS BIGINT), CAST(101 AS BIGINT), CAST(600.00 AS DECIMAL(10,2)), TIMESTAMP '2025-08-20 17:30:00'),
  (CAST(1004 AS BIGINT), CAST(101 AS BIGINT), CAST(550.00 AS DECIMAL(10,2)), TIMESTAMP '2025-10-05 16:20:00'),
  (CAST(1005 AS BIGINT), CAST(100 AS BIGINT), CAST(520.00 AS DECIMAL(10,2)), TIMESTAMP '2025-11-12 11:30:00'),
  (CAST(1006 AS BIGINT), CAST(102 AS BIGINT), CAST(1200.00 AS DECIMAL(10,2)), TIMESTAMP '2025-11-15 14:30:00'),
  (CAST(1007 AS BIGINT), CAST(102 AS BIGINT), CAST(300.00 AS DECIMAL(10,2)), TIMESTAMP '2025-12-20 10:00:00'),
  (CAST(1008 AS BIGINT), CAST(101 AS BIGINT), CAST(400.00 AS DECIMAL(10,2)), TIMESTAMP '2025-12-25 09:45:00'),
  (CAST(1009 AS BIGINT), CAST(103 AS BIGINT), CAST(850.00 AS DECIMAL(10,2)), TIMESTAMP '2026-01-10 12:00:00'),
  (CAST(1010 AS BIGINT), CAST(103 AS BIGINT), CAST(400.00 AS DECIMAL(10,2)), TIMESTAMP '2026-01-14 09:30:00');
--
CREATE TABLE home.agg_customers_obt (
  customer_id BIGINT,
  last_order_id BIGINT,
  orders BIGINT[]
);
--
INSERT INTO home.agg_customers_obt
SELECT
    order_customer_id AS customer_id,
    array_agg(order_id ORDER BY created_at DESC)[1] AS last_order_id,
    ARRAY_AGG(order_id)
FROM home.fct_orders
GROUP BY order_customer_id;


In [11]:
%%featureql --client client --hide-dataframe

SELECT
    count := CAST(4 AS INT)
;


In [12]:
%%featureql --client client --hide-dataframe

SELECT
    customer_id,
    customer_ltv := customer_id.RELATED(
        SUM(order_price)
        GROUP BY order_customer_id
    ),
FROM fm.home
FOR
    customer_id := @BIND_KEYSET(dim_customers_keyset),
    order_id := @BIND_KEYSET(fct_orders_keyset),
;


In [13]:
%%featureql --client client --hide-dataframe

WITH
    customer_orders_details := EXTEND(
        ZIP(customer_orders AS order_id)
        WITH order_price AS order_price
        VIA order_id BIND TO order_id
    )
SELECT
    customer_id,
    customer_ltv_arr := ARRAY_SUM(customer_orders_details[order_price]),
FROM fm.home
FOR
    customer_id := @BIND_KEYSET(dim_customers_keyset),
;


In [14]:
%%featureql --client client

CREATE OR REPLACE FEATURES IN FM.HOME AS
SELECT
    customer_ltv := customer_id.RELATED(
        SUM(order_price) GROUP BY order_customer_id
    ),
    -- Cents keep the promo rule BIGINT-comparable with Redis serving fields
    customer_ltv_cents := CAST(customer_ltv * 100 AS BIGINT),
    recency := DATE_DIFF(
        TIMESTAMP '2026-02-01',
        last_order_id.RELATED(order_created_at),
        'day'
    ),
    show_promocode_offline := recency > 30 AND customer_ltv_cents > 100000
;


,feature_name,status,message
0,FM.HOME.CUSTOMER_LTV,CREATED,Feature created as not exists
1,FM.HOME.CUSTOMER_LTV_CENTS,CREATED,Feature created as not exists
2,FM.HOME.RECENCY,CREATED,Feature created as not exists
3,FM.HOME.SHOW_PROMOCODE_OFFLINE,CREATED,Feature created as not exists


`show_promocode_offline` is the canonical rule. Batch jobs, hybrid SQL, and the serving `VARIANT()` all reuse it.

For FeatureQL patterns such as `RELATED()` vs SQL, or array `EXTEND()`, see the [FeatureQL homepage companion](https://featuremesh.com/docs/tutorials/homepage/featureql).

### Compute features in batch

Bind every customer and project the persisted features.


In [15]:
%%featureql --client client

SELECT
    customer_id,
    customer_ltv,
    recency,
    show_promocode_offline,
FROM fm.home
FOR
    customer_id := @BIND_KEYSET(dim_customers_keyset),
    order_id := @BIND_KEYSET(fct_orders_keyset),
;


,FM.HOME.CUSTOMER_ID,FM.HOME.CUSTOMER_LTV,FM.HOME.RECENCY,FM.HOME.SHOW_PROMOCODE_OFFLINE
0,100,1350.0,81,True
1,102,1500.0,43,True
2,101,1550.0,38,True
3,103,1250.0,18,False


The result should contain four customers: three receive the promotion and customer `103` does not.

On the homepage, the same idea appears as hybrid SQL — the warehouse keeps speaking SQL while FeatureQL owns the definitions:


In [16]:
%%featureql --client client

/* SQL */
SELECT
    show_promocode_offline,
    COUNT(1) AS num_customers
FROM FEATUREQL(
    SELECT
        customer_id,
        show_promocode_offline := fm.home.show_promocode_offline
    FROM fm.home
    FOR
        customer_id := @BIND_KEYSET(dim_customers_keyset),
        order_id := @BIND_KEYSET(fct_orders_keyset)
)
GROUP BY show_promocode_offline
;


,SHOW_PROMOCODE_OFFLINE,num_customers
0,False,1
1,True,3


One registry; many consumers (notebooks, dashboards, training jobs).

## Serving

Same decision, live path. Analytics used warehouse tables; serving reads precomputed fields from Redis. The business predicate does not change — `VARIANT()` swaps the dependencies.

This section requires the `serving` backend and Redis.

### Seed Redis

Load the precomputed fields first — each customer is a Redis hash with `days_since_order` and `lifetime_value_cents`. The next steps only *read* these keys.


In [17]:
_result = SERVING_EXECUTORS['redis'].statement("""DEL tutorial:featuremesh:100 tutorial:featuremesh:101 tutorial:featuremesh:102 tutorial:featuremesh:103
HSET tutorial:featuremesh:100 days_since_order 81 lifetime_value_cents 135000
HSET tutorial:featuremesh:101 days_since_order 38 lifetime_value_cents 155000
HSET tutorial:featuremesh:102 days_since_order 43 lifetime_value_cents 150000
HSET tutorial:featuremesh:103 days_since_order 18 lifetime_value_cents 125000
""")
if _result.errors:
    raise RuntimeError(_result.errors)


### Define serving sources

Connect Redis with `SOURCE_REDIS()`, then map hash fields with `EXTERNAL_REDIS()` into features that match the batch types (`BIGINT` recency and LTV cents).


In [18]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURES IN fm.home AS
SELECT
    redis_source := SOURCE_REDIS(
        'redis://127.0.0.1:6379'
        WITH (timeout='500ms')
    ),
    redis_key := 'tutorial:featuremesh:' || UNSAFE_CAST(customer_id AS VARCHAR),
    recency_online := CAST(
        EXTERNAL_REDIS(KEY redis_key FIELD 'days_since_order' FROM redis_source)
        AS BIGINT
    ),
    customer_ltv_cents_online := CAST(
        EXTERNAL_REDIS(KEY redis_key FIELD 'lifetime_value_cents' FROM redis_source)
        AS BIGINT
    )
;


,FEATURE_NAME,STATUS,MESSAGE
0,FM.HOME.REDIS_SOURCE,CREATED,Feature created as not exists
1,FM.HOME.REDIS_KEY,CREATED,Feature created as not exists
2,FM.HOME.RECENCY_ONLINE,CREATED,Feature created as not exists
3,FM.HOME.CUSTOMER_LTV_CENTS_ONLINE,CREATED,Feature created as not exists


### Re-use features for serving

`VARIANT()` takes `show_promocode_offline` and replaces warehouse dependencies with the Redis-backed ones — without rewriting the rule:


In [19]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURE fm.home.show_promocode_online AS
VARIANT(
    fm.home.show_promocode_offline
    REPLACING fm.home.recency, fm.home.customer_ltv_cents
    WITH fm.home.recency_online, fm.home.customer_ltv_cents_online
);


,FEATURE_NAME,STATUS,MESSAGE
0,FM.HOME.SHOW_PROMOCODE_ONLINE,CREATED,Feature created as not exists


Evaluate the serving variant:


In [20]:
%%featureql --client serving_client --hide-dataframe

REFRESH FEATURES FM.HOME.REDIS_SOURCE;


In [21]:
%%featureql --client serving_client

SELECT
    FM.HOME.CUSTOMER_ID := BIND_VALUES(ARRAY[100, 101, 102, 103]),
    FM.HOME.RECENCY_ONLINE,
    FM.HOME.CUSTOMER_LTV_CENTS_ONLINE,
    FM.HOME.SHOW_PROMOCODE_ONLINE
;


,FM.HOME.CUSTOMER_ID,FM.HOME.RECENCY_ONLINE,FM.HOME.CUSTOMER_LTV_CENTS_ONLINE,FM.HOME.SHOW_PROMOCODE_ONLINE
0,100,81,135000,True
1,101,38,155000,True
2,102,43,150000,True
3,103,18,125000,False


Customers `100`, `101`, and `102` return `true`; customer `103` returns `false`. One promo definition now runs in two execution contexts.

### Compile as prepared statement

`PREPARED_STATEMENT()` compiles the serving variant for low-latency calls — bind `customer_id`, get the boolean back.


In [22]:
%%featureql --client serving_client

CREATE OR REPLACE FEATURE fm.home.show_promocode_online_ps AS
PREPARED_STATEMENT(
    fm.home.show_promocode_online
    USING fm.home.customer_id
);


,FEATURE_NAME,STATUS,MESSAGE
0,FM.HOME.SHOW_PROMOCODE_ONLINE_PS,CREATED,Feature created as not exists


In [23]:
%%featureql --client serving_client --hide-dataframe

REFRESH FEATURES FM.HOME.SHOW_PROMOCODE_ONLINE_PS;


In [24]:
import json

_prepared_id = 'FM.HOME.SHOW_PROMOCODE_ONLINE_PS'
_prepared_calls = [
    "{\"input_table_1\": [[100], [101], [102], [103]]}",
    "{\"input_table_1\": [[100]]}",
]
for _payload in _prepared_calls:
    _inputs = json.loads(_payload) if isinstance(_payload, str) else _payload
    _result = serving_client.execute_prepared_statement(_prepared_id, _inputs)
    if _result.errors:
        raise RuntimeError(_result.errors)
    display(_result.dataframe)


,FM.HOME.CUSTOMER_ID,FM.HOME.SHOW_PROMOCODE_ONLINE_PS
0,100,True
1,101,True
2,102,True
3,103,False


,FM.HOME.CUSTOMER_ID,FM.HOME.SHOW_PROMOCODE_ONLINE_PS
0,100,True


### Integrate anywhere

Evaluation is an API call. For customer `100`:

<pre><code>POST /api/evaluate
Content-Type: application/json

&#123;
    "id": "fm.home.show_promocode_online_ps",
    "inputs": [
        ["100"]
    ]
&#125;</code></pre>

## What's next

- [Real-time segmentation](https://featuremesh.com/docs/tutorials/serving/realtime) — live Redis updates in more depth
- [Federated serving](https://featuremesh.com/docs/tutorials/serving/federation) — PostgreSQL + Redis in one serving graph
- [E-commerce](https://featuremesh.com/docs/tutorials/analytics/ecomm) — entities, `RELATED()`, and `EXTEND()`
- [SaaS metrics](https://featuremesh.com/docs/tutorials/analytics/saas) — MRR, cohort NRR, and composable customer health


## Clean up

Drop tutorial features and fixture data so you can re-run the notebook from a clean state.


In [25]:
%%featureql --client serving_client --hide-dataframe

DROP FEATURES IF EXISTS
    FM.HOME.SHOW_PROMOCODE_ONLINE_PS,
    FM.HOME.SHOW_PROMOCODE_ONLINE,
    FM.HOME.CUSTOMER_LTV_CENTS_ONLINE,
    FM.HOME.RECENCY_ONLINE,
    FM.HOME.REDIS_KEY,
    FM.HOME.REDIS_SOURCE
;


In [26]:
_result = SERVING_EXECUTORS['redis'].statement("""DEL tutorial:featuremesh:100 tutorial:featuremesh:101 tutorial:featuremesh:102 tutorial:featuremesh:103
""")
if _result.errors:
    raise RuntimeError(_result.errors)


In [27]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN fm.home UP TO LEVEL 9;


In [28]:
%%featureql --client client --hide-dataframe

DROP FEATURES IF EXISTS IN fm.home WHERE FUNCTION = 'KEYSET'


*** Warning(s) (1) ***

  1. [DROP-FEATURES-EXPRESSION-EMPTY] No features found in expression: *(FEATURES IF EXISTS IN FM.HOME WHERE FUNCTION = 'KEYSET') (acknowledge with ACK-53KA)


In [29]:
%%featureql --client client --hide-dataframe

/* SQL */
DROP TABLE IF EXISTS home.dim_customers;
--
DROP TABLE IF EXISTS home.fct_orders;
--
DROP TABLE IF EXISTS home.agg_customers_obt;


---

Source tutorial: [/docs/tutorials/homepage/featuremesh](https://featuremesh.com/docs/tutorials/homepage/featuremesh)
